In [1]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]                              # hdfs://namenode:9000
RAW = HDFS + cfg["paths"]["raw"]                     # zona raw: el CSV tal cual
CURATED = HDFS + cfg["paths"]["curated"]             # zona curada: Parquet limpio
LANG_INFO = HDFS + cfg["paths"]["language_info"]     # tabla auxiliar de idiomas
RESULTS = HDFS + cfg["paths"]["results"]             # zona de resultados

# Igual que en el Tema 5: actuamos como root en HDFS, ANTES de crear la sesión
os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-pipeline")
         .master(cfg["spark"]["master"])                             # en clase: local[*]
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [2]:
local_csv = cfg["input"]["local_csv"]
Path = spark.sparkContext._jvm.org.apache.hadoop.fs.Path
fs = Path(RAW).getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
dest = Path(RAW + "/" + os.path.basename(local_csv))

if fs.exists(dest):
    print("El CSV ya está en HDFS:", dest)
elif os.path.exists(local_csv):
    t = time.time()
    fs.mkdirs(Path(RAW))
    fs.copyFromLocalFile(False, True, Path("file://" + local_csv), dest)
    print(f"Copiado a HDFS en {time.time() - t:.0f} s:", dest)
else:
    raise FileNotFoundError("No está en HDFS ni en data/. Descarga el CSV (Paso 3) y súbelo (Paso 4).")


El CSV ya está en HDFS: hdfs://namenode:9000/raw/wikirank/wikirank_20240801.csv


In [3]:
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")

raw = (spark.read.schema(schema)
         .option("header", True)
         .option("escape", '"')          # algunos títulos llevan comillas dentro
         .csv(RAW))
raw.printSchema()
raw.show(3, truncate=40)


root
 |-- Wikipedia_article_URL: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Page_ID: long (nullable = true)
 |-- Title: string (nullable = true)
 |-- WikiRank_score: double (nullable = true)
 |-- WikiRank_URL: string (nullable = true)

+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|                   Wikipedia_article_URL|Language|Page_ID|  Title|WikiRank_score|                            WikiRank_URL|
+----------------------------------------+--------+-------+-------+--------------+----------------------------------------+
|https://ar.wikipedia.org/wiki/%D9%85%...|      ar|      7|    ماء|       85.4175|https://wikirank.net/ar/%D9%85%D8%A7%...|
|https://ar.wikipedia.org/wiki/%D8%B1%...|      ar|     38|رياضيات|        79.166|https://wikirank.net/ar/%D8%B1%D9%8A%...|
|https://ar.wikipedia.org/wiki/%D8%A5%...|      ar|     62|إستونيا|       90.6549|https://wikirank.net/ar/%D8%A5%D8%B

In [4]:
q = cfg["quality"]
t = time.time()

# Una sola pasada: total de filas, nulos por columna y scores fuera de rango
checks = raw.agg(
    F.count("*").alias("filas_total"),
    *[F.sum(F.col(c).isNull().cast("int")).alias("nulos_" + c) for c in raw.columns],
    F.sum((~F.col("WikiRank_score").between(q["score_min"], q["score_max"])).cast("int"))
     .alias("score_fuera_de_rango"),
).first().asDict()

# Duplicados: la misma clave (idioma + Page_ID) más de una vez (Tema 4: "check that the key is unique")
checks["claves_duplicadas"] = (raw.groupBy("Language", "Page_ID").count()
                                  .filter("count > 1").count())

print(f"Calidad calculada en {time.time() - t:.0f} s")
for k, v in checks.items():
    print(f"{k:30s} {v}")


Calidad calculada en 224 s
filas_total                    47019883
nulos_Wikipedia_article_URL    0
nulos_Language                 0
nulos_Page_ID                  0
nulos_Title                    0
nulos_WikiRank_score           0
nulos_WikiRank_URL             0
score_fuera_de_rango           0
claves_duplicadas              0


In [5]:
url = F.col("Wikipedia_article_URL")
curated = (raw
    .dropna(subset=["Language", "Page_ID", "WikiRank_score"])                  # filas sin un campo clave
    .filter(F.col("WikiRank_score").between(q["score_min"], q["score_max"]))  # scores imposibles
    .withColumn("Language", F.lower(F.trim("Language")))
    .withColumn("Title", F.coalesce(F.col("Title"), F.lit("")))               # título vacío en vez de nulo
    # nº de "%" en la URL = letras no latinas del título (feature para el modelo)
    .withColumn("url_pct_count",
                (F.length(url) - F.length(F.regexp_replace(url, "%", ""))).cast("int"))
    .drop("Wikipedia_article_URL", "WikiRank_URL"))                           # ver nota

if checks["claves_duplicadas"] > 0:                    # solo si la celda 4 encontró duplicados
    curated = curated.dropDuplicates(["Language", "Page_ID"])

In [6]:
part = cfg["partition_column"]
t = time.time()
(curated.repartition(part)         # un solo fichero por idioma (evita miles de ficheros pequeños)
    .write.mode("overwrite")        # overwrite: re-ejecutar no duplica (Tema 5)
    .partitionBy(part)
    .parquet(CURATED))
print(f"Escrito en {time.time() - t:.0f} s")

articles = spark.read.parquet(CURATED)
checks["filas_curadas"] = articles.count()
checks["filas_eliminadas"] = checks["filas_total"] - checks["filas_curadas"]
print("Filas curadas:", checks["filas_curadas"], "| eliminadas:", checks["filas_eliminadas"])


Escrito en 197 s
Filas curadas: 47019883 | eliminadas: 0


In [7]:
quality = spark.createDataFrame([checks])
quality.coalesce(1).write.mode("overwrite").json(RESULTS + "/quality_report")
quality.show(vertical=True)


-RECORD 0-------------------------------
 claves_duplicadas           | 0        
 filas_curadas               | 47019883 
 filas_eliminadas            | 0        
 filas_total                 | 47019883 
 nulos_Language              | 0        
 nulos_Page_ID               | 0        
 nulos_Title                 | 0        
 nulos_WikiRank_URL          | 0        
 nulos_WikiRank_score        | 0        
 nulos_Wikipedia_article_URL | 0        
 score_fuera_de_rango        | 0        



In [8]:
es = articles.filter("Language = 'es'")
es.explain()
es.count()


== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [Page_ID#111L,Title#112,WikiRank_score#113,url_pct_count#114,Language#115] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [isnotnull(Language#115), (Language#115 = es)], PushedFilters: [], ReadSchema: struct<Page_ID:bigint,Title:string,WikiRank_score:double,url_pct_count:int>




1905093